# Main Kaggle — Phase A setup và diagnostics (TV2 Qwen VI, TV4 Gemma VI)
Notebook này **không gọi model**. Nó kiểm Dataset private, giải nén vào `/kaggle/working`,
cài venv inference, ghi diagnostics GPU/Internet/Secrets và kiểm runner Phase B đã sẵn sàng chưa.
Chạy **từng cell**, không Run all. Clear outputs trước khi commit notebook.

Chuẩn bị: Settings → Accelerator = GPU, Internet = On; Add Input → Dataset private chứa
`VLM_ENVI_Main_Inference_Team_v1.zip` (SHA256 do TV5 công bố).

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROLE = "TV2"  # Kaggle roles: TV2 (qwen vi) hoặc TV4 (gemma vi)
REPO_URL = "https://github.com/MinhGioChai/cv-final-project.git"
BRANCH = "work/main-qwen-vi"  # branch chứa scripts/cloud/kaggle_main.py đã push
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
EVIDENCE = WORK / "evidence"
assert ROLE in {"TV2", "TV4"}, "Notebook Kaggle chỉ dành cho TV2/TV4"


def run(cmd, cwd=None):
    """Run a command, show stdout, stop with stderr on failure."""
    done = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True)
    print(done.stdout)
    if done.returncode:
        raise RuntimeError(done.stderr.strip()[-3000:] or f"exit {done.returncode}")
    return done.stdout

## 1. Lấy công cụ Kaggle từ Git
Clone đúng branch vào `/kaggle/working` (cần Internet). Commit được ghi lại làm provenance.

In [ ]:
CLONE = WORK / "cv-final-project"
if not CLONE.exists():
    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE])
TOOL_COMMIT = run(["git", "rev-parse", "HEAD"], cwd=CLONE).strip()
TOOL = CLONE / "scripts/cloud/kaggle_main.py"
assert TOOL.is_file(), "Branch chưa có scripts/cloud/kaggle_main.py; kiểm BRANCH"
print("tool commit:", TOOL_COMMIT)

## 2. Tìm và kiểm bundle trong `/kaggle/input`
`locate` từ chối nếu không có hoặc có nhiều bundle (nhiều Dataset/version). Nếu Kaggle đã tự giải nén
ZIP thành thư mục, `prepare` dừng lại: hỏi TV5 rồi mới thêm `--allow-extracted`.

In [ ]:
_ = run([sys.executable, TOOL, "locate", "--role", ROLE])

In [ ]:
BUNDLE_ROOT = WORK / "VLM_ENVI_Main_Inference_Team_v1"
if BUNDLE_ROOT.exists():
    print("Dùng lại workspace đã kiểm:", BUNDLE_ROOT)
else:
    prepared = json.loads(run([sys.executable, TOOL, "prepare", "--role", ROLE,
        "--report", EVIDENCE / f"{ROLE}_prepare.json"]))
    assert prepared["working_copy_verified"]
PROJECT = BUNDLE_ROOT / "project"
assert (PROJECT / "scripts/setup_inference.py").is_file()

## 3. Cài môi trường inference riêng
Dùng `scripts/setup_inference.py` với pins hiện có (torch 2.7.1 cu126). Không pip upgrade hoặc đổi precision.

In [ ]:
INSTALL = False  # đặt True cho lần cài đầu của phiên
ENV = PROJECT / ".venv-cloud"
if INSTALL:
    run([sys.executable, "scripts/setup_inference.py", "--env", ENV], cwd=PROJECT)
PYTHON = ENV / "bin/python"
assert PYTHON.is_file(), "Chưa có .venv-cloud: đặt INSTALL=True rồi chạy lại cell"
_ = run([PYTHON, "-m", "pip", "check"], cwd=PROJECT)

## 4. Secrets (chỉ TV4/Gemma)
Qwen là model public: TV2 không cần `HF_TOKEN`. TV4 thêm secret `HF_TOKEN` trong Add-ons → Secrets
và bật cho notebook. Không print token.

In [ ]:
if ROLE == "TV4":
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 5. Diagnostics thật (không load model)
Ghi GPU/VRAM/compute capability, package fingerprint, Internet và secret có hay không (không ghi giá trị).
Report JSON nằm trong `/kaggle/working/evidence/`; mỗi lần chạy dùng tên mới.

In [ ]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
_ = run([sys.executable, TOOL, "diagnose", "--role", ROLE, "--inference-python", PYTHON,
     "--project", PROJECT, "--check-internet", "--report", EVIDENCE / f"{ROLE}_diagnose_{stamp}.json"])

## 6. Restore ảnh và plan shard
Restore kiểm 783 JPEG + source inputs theo manifest đã bind; file khác bytes bị từ chối.
Plan đúng: model/language của role, 1.000 cells, `phase_b_execution_ready=false`.

In [ ]:
run([PYTHON, "scripts/main_team_bundle.py", "restore-assets",
     "--archive", BUNDLE_ROOT / "main_assets/main_assets_v1.zip", "--root", PROJECT], cwd=PROJECT)
_ = run([PYTHON, "scripts/team_assignment.py", "plan",
     "--assignment", f"configs/team_assignments_v1/{ROLE}.yaml"], cwd=PROJECT)

## 7. Runner Phase B đã sẵn sàng chưa?
In lệnh smoke mục tiêu và các blocker (TV1 `--language`, config freeze v2, approval). Không chạy lệnh.

In [ ]:
_ = run([sys.executable, TOOL, "runner-check", "--role", ROLE, "--project", PROJECT,
     "--inference-python", PYTHON])

## Phase B — chưa bật
Sau khi các PR được tích hợp, TV5 chốt commit/release chung, protocol v2 và execution approval.
Khi đó cell dưới được thay bằng workflow release đã kiểm; không bật bằng cách đổi boolean.
Backup: tải `/kaggle/working/evidence/` và outputs, hoặc Save Version rồi thử restore.

In [ ]:
PHASE_B_APPROVED = False
if PHASE_B_APPROVED:
    raise RuntimeError("TV5 phải thay cell này bằng verified shared release workflow sau PR integration.")
print("Phase A only: no inference calls.")